# Solar Panel Fault Detection - Exploratory Data Analysis

This notebook performs comprehensive exploratory data analysis on the solar panel fault detection dataset.

## Contents
1. [Setup and Imports](#1-setup-and-imports)
2. [Dataset Overview](#2-dataset-overview)
3. [Class Distribution Analysis](#3-class-distribution-analysis)
4. [Image Statistics](#4-image-statistics)
5. [Sample Visualization](#5-sample-visualization)
6. [Data Augmentation](#6-data-augmentation)
7. [Correlation Analysis](#7-correlation-analysis)
8. [Conclusions](#8-conclusions)

## 1. Setup and Imports

In [ ]:
# Standard library imports
import os
import sys
from pathlib import Path
from collections import Counter

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

# Data processing
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Computer Vision
import cv2

# Configuration
from src.utils import load_config, get_class_distribution, print_class_distribution

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

%matplotlib inline

print("✅ Setup complete!")

In [ ]:
# Load configuration
config = load_config()

# Dataset paths
DATA_DIR = Path(config['dataset']['processed_dir'])
TRAIN_DIR = DATA_DIR / 'train'
VAL_DIR = DATA_DIR / 'val'
TEST_DIR = DATA_DIR / 'test'

# Class names
CLASSES = config['model']['classes']

print(f"Data Directory: {DATA_DIR}")
print(f"Classes: {CLASSES}")

## 2. Dataset Overview

In [ ]:
def count_images(directory):
    """Count images in directory and subdirectories."""
    counts = {}
    total = 0
    
    if not directory.exists():
        return counts, total
    
    for class_dir in directory.iterdir():
        if class_dir.is_dir():
            images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
            counts[class_dir.name] = len(images)
            total += len(images)
    
    return counts, total

# Count images in each split
train_counts, train_total = count_images(TRAIN_DIR)
val_counts, val_total = count_images(VAL_DIR)
test_counts, test_total = count_images(TEST_DIR)

total_images = train_total + val_total + test_total

print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"\nTotal Images: {total_images}")
print(f"  - Training: {train_total} ({train_total/total_images*100:.1f}%)")
print(f"  - Validation: {val_total} ({val_total/total_images*100:.1f}%)")
print(f"  - Test: {test_total} ({test_total/total_images*100:.1f}%)")
print("\nClass Names:")
for i, cls in enumerate(CLASSES, 1):
    print(f"  {i}. {cls}")

## 3. Class Distribution Analysis

In [ ]:
# Create class distribution DataFrame
distribution_data = []

for split_name, counts in [('Train', train_counts), ('Val', val_counts), ('Test', test_counts)]:
    for class_name, count in counts.items():
        distribution_data.append({
            'Split': split_name,
            'Class': class_name,
            'Count': count
        })

df_distribution = pd.DataFrame(distribution_data)

# Pivot for better visualization
df_pivot = df_distribution.pivot(index='Class', columns='Split', values='Count').fillna(0)
df_pivot['Total'] = df_pivot.sum(axis=1)
df_pivot = df_pivot.reindex(CLASSES)

print("Class Distribution:")
print(df_pivot)

# Calculate percentages
df_pivot['Percentage'] = (df_pivot['Total'] / df_pivot['Total'].sum() * 100).round(2)
print("\nWith Percentages:")
print(df_pivot)

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Stacked bar chart
df_pivot[['Train', 'Val', 'Test']].plot(
    kind='bar', 
    stacked=True, 
    ax=axes[0],
    color=['#3498db', '#e74c3c', '#2ecc71'],
    alpha=0.8
)
axes[0].set_title('Class Distribution by Split', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class', fontsize=12)
axes[0].set_ylabel('Number of Images', fontsize=12)
axes[0].legend(title='Split')
axes[0].tick_params(axis='x', rotation=45)

# Pie chart
colors = ['#2ecc71', '#e74c3c', '#e67e22', '#f1c40f']
axes[1].pie(
    df_pivot['Total'], 
    labels=df_pivot.index, 
    autopct='%1.1f%%',
    colors=colors,
    startangle=90
)
axes[1].set_title('Overall Class Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

### Class Imbalance Analysis

In [ ]:
# Calculate imbalance metrics
class_counts = df_pivot['Total'].values
max_count = class_counts.max()
min_count = class_counts.min()
imbalance_ratio = max_count / min_count

print("="*60)
print("CLASS IMBALANCE ANALYSIS")
print("="*60)
print(f"\nMost frequent class: {df_pivot['Total'].idxmax()} ({max_count} images)")
print(f"Least frequent class: {df_pivot['Total'].idxmin()} ({min_count} images)")
print(f"Imbalance Ratio: {imbalance_ratio:.2f}:1")

if imbalance_ratio > 3:
    print("\n⚠️ WARNING: Significant class imbalance detected!")
    print("Consider using:")
    print("  - Weighted loss function")
    print("  - Data augmentation for minority classes")
    print("  - Oversampling/Undersampling techniques")
else:
    print("\n✅ Class distribution is relatively balanced")

## 4. Image Statistics

In [ ]:
def analyze_image_properties(image_path):
    """Analyze properties of a single image."""
    try:
        with Image.open(image_path) as img:
            return {
                'width': img.width,
                'height': img.height,
                'mode': img.mode,
                'format': img.format,
                'aspect_ratio': img.width / img.height
            }
    except Exception as e:
        return None

# Collect image statistics
image_stats = []

for split_dir in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if not split_dir.exists():
        continue
    
    for class_dir in split_dir.iterdir():
        if not class_dir.is_dir():
            continue
        
        for img_path in list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')):
            stats = analyze_image_properties(img_path)
            if stats:
                stats['class'] = class_dir.name
                stats['split'] = split_dir.name
                stats['path'] = str(img_path)
                image_stats.append(stats)

df_stats = pd.DataFrame(image_stats)

print(f"Analyzed {len(df_stats)} images")
print("\nImage Statistics Summary:")
print(df_stats[['width', 'height', 'aspect_ratio']].describe())

In [ ]:
# Visualize image dimensions
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Width distribution
axes[0, 0].hist(df_stats['width'], bins=30, color='#3498db', alpha=0.7, edgecolor='black')
axes[0, 0].set_title('Image Width Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Width (pixels)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df_stats['width'].mean(), color='red', linestyle='--', 
                   label=f"Mean: {df_stats['width'].mean():.0f}")
axes[0, 0].legend()

# Height distribution
axes[0, 1].hist(df_stats['height'], bins=30, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[0, 1].set_title('Image Height Distribution', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Height (pixels)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(df_stats['height'].mean(), color='red', linestyle='--',
                   label=f"Mean: {df_stats['height'].mean():.0f}")
axes[0, 1].legend()

# Aspect ratio distribution
axes[1, 0].hist(df_stats['aspect_ratio'], bins=30, color='#2ecc71', alpha=0.7, edgecolor='black')
axes[1, 0].set_title('Aspect Ratio Distribution', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Aspect Ratio (width/height)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(df_stats['aspect_ratio'].mean(), color='red', linestyle='--',
                   label=f"Mean: {df_stats['aspect_ratio'].mean():.2f}")
axes[1, 0].legend()

# Scatter plot of width vs height
scatter = axes[1, 1].scatter(df_stats['width'], df_stats['height'], 
                            c=df_stats['class'].astype('category').cat.codes,
                            cmap='viridis', alpha=0.5, s=20)
axes[1, 1].set_title('Image Dimensions Scatter Plot', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Width (pixels)')
axes[1, 1].set_ylabel('Height (pixels)')

plt.tight_layout()
plt.savefig('../results/image_statistics.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Sample Visualization

In [ ]:
def display_sample_images(data_dir, class_name, num_samples=5):
    """Display sample images from a specific class."""
    class_dir = data_dir / class_name
    if not class_dir.exists():
        return []
    
    images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
    return images[:num_samples]

# Display samples from each class
fig, axes = plt.subplots(len(CLASSES), 5, figsize=(20, 4*len(CLASSES)))

for i, class_name in enumerate(CLASSES):
    sample_images = display_sample_images(TRAIN_DIR, class_name, 5)
    
    for j in range(5):
        if j < len(sample_images):
            img = Image.open(sample_images[j])
            axes[i, j].imshow(img)
            axes[i, j].axis('off')
        else:
            axes[i, j].axis('off')
        
        if j == 0:
            axes[i, j].set_ylabel(class_name.upper(), fontsize=12, fontweight='bold', rotation=0, 
                                 ha='right', va='center')

plt.suptitle('Sample Images from Each Class', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('../results/sample_images.png', dpi=300, bbox_inches='tight')
plt.show()

### Pixel Intensity Analysis

In [ ]:
def analyze_pixel_intensities(image_paths, max_images=100):
    """Analyze pixel intensity distributions."""
    intensities = []
    
    for img_path in image_paths[:max_images]:
        img = cv2.imread(str(img_path))
        if img is not None:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            intensities.extend(gray.flatten())
    
    return intensities

# Analyze pixel intensities by class
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

colors = ['#2ecc71', '#e74c3c', '#e67e22', '#f1c40f']

for i, class_name in enumerate(CLASSES):
    class_dir = TRAIN_DIR / class_name
    if class_dir.exists():
        images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
        intensities = analyze_pixel_intensities(images, 50)
        
        axes[i].hist(intensities, bins=50, color=colors[i], alpha=0.7, edgecolor='black')
        axes[i].set_title(f'{class_name.upper()} - Pixel Intensity', fontweight='bold')
        axes[i].set_xlabel('Pixel Intensity')
        axes[i].set_ylabel('Frequency')
        axes[i].axvline(np.mean(intensities), color='red', linestyle='--',
                       label=f"Mean: {np.mean(intensities):.1f}")
        axes[i].legend()

plt.suptitle('Pixel Intensity Distribution by Class', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/pixel_intensities.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Data Augmentation

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Define augmentation pipeline
augmentation_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.Blur(blur_limit=3, p=0.2),
])

# Get a sample image
sample_class = CLASSES[0]
sample_dir = TRAIN_DIR / sample_class
sample_images = list(sample_dir.glob('*.jpg'))

if sample_images:
    sample_img = cv2.imread(str(sample_images[0]))
    sample_img = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)
    
    # Apply augmentations
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    
    # Original image
    axes[0, 0].imshow(sample_img)
    axes[0, 0].set_title('Original', fontweight='bold')
    axes[0, 0].axis('off')
    
    # Augmented versions
    for i in range(7):
        augmented = augmentation_pipeline(image=sample_img)
        aug_img = augmented['image']
        
        row = (i + 1) // 4
        col = (i + 1) % 4
        axes[row, col].imshow(aug_img)
        axes[row, col].set_title(f'Augmentation {i+1}', fontweight='bold')
        axes[row, col].axis('off')
    
    plt.suptitle('Data Augmentation Examples', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/augmentation_examples.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No sample images found for augmentation demo")

## 7. Correlation Analysis

In [ ]:
# Analyze correlation between image properties
if len(df_stats) > 0:
    # Create correlation matrix
    corr_data = df_stats[['width', 'height', 'aspect_ratio']].corr()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr_data, annot=True, cmap='coolwarm', center=0,
                square=True, linewidths=1, cbar_kws={"shrink": .8}, ax=ax)
    ax.set_title('Correlation Matrix: Image Properties', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/correlation_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nCorrelation Matrix:")
    print(corr_data)

## 8. Conclusions

In [ ]:
# Generate summary report
print("="*70)
print("EXPLORATORY DATA ANALYSIS SUMMARY")
print("="*70)

print(f"\n📊 Dataset Statistics:")
print(f"   • Total Images: {total_images}")
print(f"   • Training Set: {train_total} images ({train_total/total_images*100:.1f}%)")
print(f"   • Validation Set: {val_total} images ({val_total/total_images*100:.1f}%)")
print(f"   • Test Set: {test_total} images ({test_total/total_images*100:.1f}%)")

print(f"\n📈 Class Distribution:")
for class_name in CLASSES:
    count = df_pivot.loc[class_name, 'Total']
    percentage = df_pivot.loc[class_name, 'Percentage']
    print(f"   • {class_name.capitalize()}: {count} images ({percentage}%)")

print(f"\n⚖️  Class Balance:")
print(f"   • Imbalance Ratio: {imbalance_ratio:.2f}:1")
if imbalance_ratio > 3:
    print(f"   • Status: ⚠️ Significant imbalance detected")
else:
    print(f"   • Status: ✅ Relatively balanced")

if len(df_stats) > 0:
    print(f"\n🖼️  Image Properties:")
    print(f"   • Average Width: {df_stats['width'].mean():.0f} pixels")
    print(f"   • Average Height: {df_stats['height'].mean():.0f} pixels")
    print(f"   • Average Aspect Ratio: {df_stats['aspect_ratio'].mean():.2f}")
    print(f"   • Most Common Format: {df_stats['mode'].mode()[0]}")

print(f"\n💡 Recommendations:")
print(f"   1. Use data augmentation to address class imbalance")
print(f"   2. Resize images to {config['image']['width']}x{config['image']['height']} for model input")
print(f"   3. Apply normalization with mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]")
print(f"   4. Consider weighted loss function for training")
print(f"   5. Use stratified sampling for train/val/test splits")

print("\n" + "="*70)

---

## Next Steps

1. **Data Preprocessing**: Run `src/preprocess.py` to prepare the dataset
2. **Model Training**: Run `src/train.py` to train the model
3. **Evaluation**: Run `src/evaluate.py` to evaluate model performance
4. **Inference**: Use `src/predict.py` for making predictions

For more information, see the project README.md file.